# Stage 1 · Minimal RAG

Goal: answer questions about **a single document**, without a vector database.

Blocks:
1. Read the PDF (page by page) and clean repeated text
2. Split it into chunks, keeping track of the source page
3. Compute the embeddings with Gemini (and cache them on disk)
4. Retrieve the closest chunks and answer

**Legend for the code comments**
- `SAME`: identical to the course code
- `ADAPTED`: based on the course code, with changes (explained in the comment)
- `NEW`: written for this project, not in the course

In [ ]:
import json
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pypdf
from dotenv import load_dotenv
from google import genai
from google.genai import errors
from google.genai.types import Content, Part

load_dotenv()            # reads GEMINI_API_KEY from the .env file
client = genai.Client()

EMBEDDING_MODEL = "gemini-embedding-2"
# First choice, then fallbacks (in class these models were sometimes "under high demand")
GENERATION_MODELS = ["gemini-3.6-flash", "gemini-3.8-flash", "gemini-3.5-flash-lite"]
OLLAMA_MODEL = "qwen3.5:0.8b"   # local last resort (see `ollama list` for the models you have)

## Block 1 · Read the PDF

Unlike the course's `read_pdf` (which joins all the text into one string), we keep **each page separately**. Later, the assistant will be able to say *"this comes from page 4"*.

In [ ]:
# ADAPTED from the course (first_rag.ipynb, read_pdf): same pypdf reading,
# but we keep each page separately instead of joining all the text.
def read_pdf(path):
    """Return a list of dicts, one per page, with its page number and text."""
    reader = pypdf.PdfReader(path)
    pages = []
    for page_number, page in enumerate(reader.pages, start=1):  # start=1 so the first page is 1, not 0
        text = page.extract_text() or ""                          # some pages (e.g. image only) have no text
        pages.append({"page": page_number, "text": text})
    return pages

In [ ]:
PDF_PATH = Path("data/rag_papers/lec1_slides.pdf")   # change this path to your document

pages = read_pdf(PDF_PATH)
print("Pages:", len(pages))
print("Total characters:", sum(len(p["text"]) for p in pages))
print("\n--- Page 4 (first 300 characters) ---")
print(pages[3]["text"][:300])

### Block 1b · Clean repeated text

Slides often repeat the same line on every page (a copyright notice, the course name, the page number). That text adds noise: it makes all chunks look a bit alike.

Instead of writing the exact sentence to remove (which would only work for this course), we find it **automatically**: any line that appears on at least half of the pages is treated as a header/footer and removed. Lines that are only a number (page numbers) are removed too. This way the same code also works for other courses.

In [ ]:
# NEW (not in the course): automatic removal of headers/footers.
def clean_pages(pages, min_share=0.5):
    """Remove lines repeated on many pages (headers/footers) and lines that are only a page number."""
    line_counts = Counter()
    for p in pages:
        unique_lines = {line.strip() for line in p["text"].splitlines() if line.strip()}
        line_counts.update(unique_lines)

    threshold = max(2, int(len(pages) * min_share))
    repeated = {line for line, count in line_counts.items() if count >= threshold}

    cleaned = []
    for p in pages:
        kept = [
            line for line in p["text"].splitlines()
            if line.strip() and line.strip() not in repeated and not line.strip().isdigit()
        ]
        cleaned.append({"page": p["page"], "text": "\n".join(kept)})
    return cleaned, repeated

In [ ]:
clean, removed_lines = clean_pages(pages)

print("Removed lines (repeated on many pages):")
for line in removed_lines:
    print("  -", line)
print("\nCharacters before:", sum(len(p["text"]) for p in pages))
print("Characters after: ", sum(len(p["text"]) for p in clean))
print("\n--- Page 4 after cleaning ---")
print(clean[3]["text"][:300])

## Block 2 · Split into chunks

Same idea as the course's `naive_chunks`: windows of `chunk_size` characters that move forward by `chunk_size - overlap`.

The difference: we split **page by page** and attach a *label* to each chunk (document, page, chunk number), like writing on the back of each index card where it came from.

In [ ]:
# ADAPTED from the course (first_rag.ipynb, naive_chunks): same sliding window and same
# overlap check. NEW: we split page by page, attach a label (document, pages, chunk_num)
# and skip exact duplicates (e.g. an agenda slide repeated in every section).
def split_into_chunks(pages, document_name, chunk_size=1000, overlap=200):
    if not 0 <= overlap < chunk_size:
        raise ValueError("overlap must be >= 0 and smaller than chunk_size")

    step = chunk_size - overlap
    chunks = []
    seen = {}                                    # chunk text -> its position in `chunks`
    for p in pages:
        text = p["text"]
        for start in range(0, len(text), step):
            piece = text[start:start + chunk_size]
            if len(piece.strip()) < 50:          # skip almost empty leftovers at the end of a page
                continue
            key = piece.strip()
            if key in seen:                      # same text already seen: keep one copy, add the page
                pages_of_copy = chunks[seen[key]]["pages"]
                if p["page"] not in pages_of_copy:
                    pages_of_copy.append(p["page"])
            else:
                seen[key] = len(chunks)
                chunks.append({
                    "document": document_name,
                    "pages": [p["page"]],
                    "chunk_num": len(chunks),
                    "text": piece,
                })
            if start + chunk_size >= len(text):  # reached the end of the page
                break
    return chunks


def pages_label(chunk):
    """'p. 4', or 'p. 2, 3, 11' for a chunk found on several pages."""
    return "p. " + ", ".join(str(n) for n in chunk["pages"])

In [ ]:
chunks = split_into_chunks(clean, document_name=PDF_PATH.stem)

print("Number of chunks:", len(chunks))
print("Chunks found on several pages (kept once):")
for c in chunks:
    if len(c["pages"]) > 1:
        print(f"  {pages_label(c)} | {c['text'][:60]!r}")

example = chunks[5]
print(f"\n--- Chunk {example['chunk_num']} | {example['document']} | {pages_label(example)} ---")
print(example["text"])

## Block 3 · Embeddings

Each chunk becomes a vector (its "address in the supermarket").

Two practical details:
- **Batches**: we send 20 chunks per request instead of one at a time (fewer trips). If Gemini answers with an error (for example the free-tier rate limit), we wait and retry, like in the course code.
- **Cache**: the embeddings are saved in `data/` (`.npy` for the vectors, `.json` for the chunks). Next time they are loaded from disk in a second, without calling the API again. If you change the chunks (size, overlap, cleaning), the cache is detected as outdated and recomputed.

In [ ]:
# ADAPTED from the course (weaviate_tutorial.ipynb, movie embeddings): same batches of 20,
# same Content/Part objects, same check on the number of embeddings, same wait-and-retry.
# NEW: a maximum number of retries, and no retry when the error is ours (ValueError).
BATCH_SIZE = 20
RETRY_DELAY_SECONDS = 30
MAX_RETRIES = 5


def embed_texts(texts):
    """Return a (n_texts, dim) float32 array with one embedding per text."""
    all_vectors = []
    for start in range(0, len(texts), BATCH_SIZE):
        batch = texts[start:start + BATCH_SIZE]
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                # One Content per text: if we pass a plain list of strings, Gemini treats it
                # as ONE content made of many parts and returns a single embedding.
                contents = [Content(parts=[Part.from_text(text=t)]) for t in batch]
                response = client.models.embed_content(model=EMBEDDING_MODEL, contents=contents)
                vectors = [e.values for e in response.embeddings]
                if len(vectors) != len(batch):
                    raise ValueError(f"Expected {len(batch)} embeddings, got {len(vectors)}")
                all_vectors.extend(vectors)
                break
            except ValueError:
                raise                      # our own check failed: retrying would give the same result
            except Exception as error:
                if attempt == MAX_RETRIES:
                    raise
                print(f"Batch {start}: {error}. Retry {attempt}/{MAX_RETRIES} in {RETRY_DELAY_SECONDS}s")
                time.sleep(RETRY_DELAY_SECONDS)
        print(f"Embedded {min(start + BATCH_SIZE, len(texts))}/{len(texts)}")
    return np.array(all_vectors, dtype=np.float32)


# NEW (the course uses np.save/np.load by hand): automatic cache that detects outdated files.
def load_or_compute_embeddings(chunks, name):
    """Load cached embeddings if they match the current chunks, otherwise compute and save them."""
    vectors_path = Path("data") / f"{name}_embeddings.npy"
    chunks_path = Path("data") / f"{name}_chunks.json"

    if vectors_path.exists() and chunks_path.exists():
        cached_chunks = json.loads(chunks_path.read_text(encoding="utf-8"))
        if cached_chunks == chunks:
            print("Loaded embeddings from cache")
            return np.load(vectors_path)
        print("Chunks changed: recomputing embeddings")

    vectors = embed_texts([c["text"] for c in chunks])
    np.save(vectors_path, vectors)
    chunks_path.write_text(json.dumps(chunks, ensure_ascii=False), encoding="utf-8")
    return vectors

In [ ]:
embeddings = load_or_compute_embeddings(chunks, name=PDF_PATH.stem)

print("Shape:", embeddings.shape)   # (number of chunks, numbers per embedding)
print("First 5 numbers of chunk 0:", embeddings[0][:5])

## Block 4 · Retrieve and answer

1. The question gets its own embedding (with the **same model** as the chunks).
2. **Cosine similarity** with every chunk: the same formula the professor wrote in class.
3. The `top_k` closest chunks are given to the LLM, labelled with document and page, with the instruction to answer **only** from them and to cite the sources.

In [ ]:
# SAME formula as the course (image search in class, 23 Sept): cosine similarity + argsort.
def search(question, top_k=5):
    """Return the top_k chunks most similar to the question, with their cosine similarity."""
    question_vector = embed_texts([question])[0]

    similarities = (embeddings @ question_vector) / (
        np.linalg.norm(embeddings, axis=1) * np.linalg.norm(question_vector)
    )
    best = np.argsort(-similarities)[:top_k]       # indices sorted from most to least similar
    return [(float(similarities[i]), chunks[i]) for i in best]

In [ ]:
results = search("What does the course say about why we should study data science?")

for score, chunk in results:
    print(f"{score:.3f} | {chunk['document']} {pages_label(chunk)} | {chunk['text'][:80]!r}")

In [ ]:
# ADAPTED from the course (first_rag.ipynb, TEMPLATE_PROMPT): same rules (only the sources,
# say when the answer is missing, cite the source). NEW: cite the page, answer in the question's language.
PROMPT_TEMPLATE = """You are a teaching assistant for a course.
Answer the question using ONLY the information in the sources below.
If the sources do not contain the answer, say clearly that the course materials do not cover it.
After each statement, cite its source in the form [document, p. N].
Answer in the same language as the question.

SOURCES:
{sources}

QUESTION: {question}
"""


# ADAPTED from the course (first_rag.ipynb, answer): same structure. NEW: numpy search instead of
# Weaviate, and a fallback between models (same idea as the LiteLLM Router seen in class).
def generate_with_fallback(prompt, models=GENERATION_MODELS, attempts_per_model=3, wait_seconds=10):
    """Try each model in order; on a server error (5xx, e.g. 'high demand') wait and retry."""
    for model in models:
        for attempt in range(1, attempts_per_model + 1):
            try:
                response = client.models.generate_content(model=model, contents=prompt)
                return response.text, model
            except errors.ServerError as error:
                print(f"{model}: {error.code} (attempt {attempt}/{attempts_per_model})")
                time.sleep(wait_seconds * attempt)   # wait a bit longer each time
            except errors.ClientError as error:
                if error.code != 404:                # 404 = model name not available for this key
                    raise                            # other 4xx (e.g. wrong key) are our problem: stop
                print(f"{model}: not available ({error.code}), skipping it")
                break
        print(f"{model} unavailable, trying the next model")

    # Last resort: a local model with Ollama (same call as the course's ollama_api_tutorial.ipynb).
    # It runs on this laptop, so it never suffers from "high demand" and works offline.
    print(f"All Gemini models busy: answering locally with Ollama ({OLLAMA_MODEL})")
    import ollama                                    # imported here: only needed when Gemini fails
    response = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[{"role": "user", "content": prompt}],
        think=False,
    )
    return response.message.content, f"ollama/{OLLAMA_MODEL}"


def answer(question, top_k=5):
    results = search(question, top_k)
    sources = "\n\n".join(
        f"[{chunk['document']}, {pages_label(chunk)}]\n{chunk['text']}" for _, chunk in results
    )
    prompt = PROMPT_TEMPLATE.format(sources=sources, question=question)
    text, model_used = generate_with_fallback(prompt)
    print(f"(answered by {model_used})")
    return text, results

In [ ]:
text, used = answer("¿Por qué debería estudiar ciencia de datos según el curso?")
print(text)
print("\n--- Sources used ---")
for score, chunk in used:
    print(f"{score:.3f} | {chunk['document']} {pages_label(chunk)}")

### Things to try

- Ask something the lecture does **not** cover (e.g. *"How do I bake a lasagna?"*): does it admit it doesn't know?
- Ask the same question in Spanish, Italian and English: do the retrieved pages change?
- Change `top_k` to 1 and to 10: how does the answer change?